Step 1: Environment Setup and Data Loading

In [ ]:
# Dataset conversion step (archived for reference).
# The private original spreadsheet is not distributed.
# Set path to your authorized source data before running.
import pandas as pd
from pathlib import Path
source_path = Path("data/hackathon_dataset.xlsx")
df = pd.read_excel(source_path)
df.to_csv("data/hackathon_dataset.csv", index=False)
print("Converted authorized source dataset to CSV")

In [ ]:
# Install required packages

# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from codecarbon import EmissionsTracker

# Start tracking carbon emissions
tracker = EmissionsTracker()
tracker.start()

# Load the labeled dataset
data = pd.read_csv('data/hackathon_dataset.csv')

# Display dataset info
print(f"Dataset shape: {data.shape}")
print(f"Columns: {data.columns.tolist()}")


Step 2: Exploratory Data Analysis

In [ ]:
# Identify text column and label columns
text_column = 'text_raw'  # Adjust based on actual column name
label_columns = [col for col in data.columns if col != text_column]

# Check label distribution
label_counts = data[label_columns].sum().sort_values(ascending=False)
print("Label counts:")
print(label_counts)

# Visualize label distribution
plt.figure(figsize=(14, 6))
label_counts.plot(kind='bar')
plt.title('Distribution of Skin Condition Labels')
plt.xlabel('Skin Condition')
plt.ylabel('Count')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# Analyze text length
data['text_length'] = data[text_column].apply(len)
print(f"Average text length: {data['text_length'].mean()}")
print(f"Min length: {data['text_length'].min()}, Max length: {data['text_length'].max()}")

# Check label correlations
plt.figure(figsize=(10, 8))
correlation_matrix = data[label_columns].corr()
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm')
plt.title('Correlation Between Skin Condition Labels')
plt.tight_layout()
plt.show()


Step 3: Text Preprocessing

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')

def preprocess_text(text):
    """Clean and preprocess text data"""
    # Convert to lowercase
    text = text.lower()
    # Remove special characters and numbers
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Remove stopwords and lemmatize (optional - may keep domain-specific terms)
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    text = ' '.join(words)
    
    return text

# Apply preprocessing
data['processed_text'] = data[text_column].apply(preprocess_text)


Step 4: Extend Dataset Using LLM for Labeling

In [ ]:
from transformers import pipeline
from tqdm import tqdm
import torch

# Load the unlabeled dataset (assuming it's available)
try:
    unlabeled_data = pd.read_csv('unlabeled_descriptions.csv')
    # Preprocess unlabeled text
    unlabeled_data['processed_text'] = unlabeled_data[text_column].apply(preprocess_text)
    
    # Function to use a pretrained model for zero-shot classification
    def label_with_llm(description, label_list):
        """Use a zero-shot classification model to label descriptions"""
        classifier = pipeline("zero-shot-classification", 
                             model="facebook/bart-large-mnli",
                             device=0 if torch.cuda.is_available() else -1)
        
        result = classifier(
            description,
            label_list,
            multi_label=True
        )
        
        # Convert scores to binary labels (threshold can be adjusted)
        threshold = 0.5
        labels = {label: 1 if score > threshold else 0 
                 for label, score in zip(result['labels'], result['scores'])}
        
        return labels
    
    # Process a sample of unlabeled data (for demonstration)
    sample_size = min(100, len(unlabeled_data))
    sample_unlabeled = unlabeled_data.sample(sample_size)
    
    # Generate labels using LLM
    print(f"Labeling {sample_size} samples with LLM...")
    new_labels = []
    for text in tqdm(sample_unlabeled['processed_text']):
        labels = label_with_llm(text, label_columns)
        new_labels.append(labels)
    
    # Convert new labels to DataFrame
    new_labeled_data = pd.DataFrame(new_labels)
    new_labeled_data[text_column] = sample_unlabeled[text_column].values
    new_labeled_data['processed_text'] = sample_unlabeled['processed_text'].values
    
    # Combine with original labeled data
    combined_data = pd.concat([data, new_labeled_data])
    print(f"Combined dataset shape: {combined_data.shape}")
    
except FileNotFoundError:
    print("Unlabeled data file not found. Proceeding with labeled data only.")
    combined_data = data.copy()


Step 5: Feature Engineering (Lightweight Approach)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2

# Create TF-IDF features with a limited number of features
tfidf_vectorizer = TfidfVectorizer(
    max_features=1500,  # Limit features for lightweight model
    ngram_range=(1, 2),  # Use unigrams and bigrams
    min_df=2,           # Minimum document frequency
    max_df=0.9          # Maximum document frequency
)

# Transform text data to TF-IDF features
X = tfidf_vectorizer.fit_transform(combined_data['processed_text'])
y = combined_data[label_columns].values

print(f"Feature matrix shape: {X.shape}")

# Feature selection to further reduce dimensionality
k = min(1000, X.shape[1])  # Select top features
selector = SelectKBest(chi2, k=k)
X_selected = selector.fit_transform(X, y.sum(axis=1))

print(f"Reduced feature matrix shape: {X_selected.shape}")

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42
)


Step 6: Model Selection and Training

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score, f1_score, hamming_loss

# Choose a lightweight model (Logistic Regression)
# Track carbon for model training
model_tracker = EmissionsTracker()
model_tracker.start()

# Define base model with L1 regularization for sparsity (lightweight)
base_model = LogisticRegression(
    C=0.1,              # Stronger regularization
    penalty='l1',       # L1 regularization for sparsity
    solver='liblinear', # Efficient solver for L1
    max_iter=1000       # Ensure convergence
)

# Create multi-label model
multi_model = MultiOutputClassifier(base_model)

# Train model
multi_model.fit(X_train, y_train)

# Make predictions
y_pred = multi_model.predict(X_test)

# Calculate metrics
accuracy = accuracy_score(y_test.reshape(-1), y_pred.reshape(-1))
f1_macro = f1_score(y_test, y_pred, average='macro')
f1_micro = f1_score(y_test, y_pred, average='micro')
hl = hamming_loss(y_test, y_pred)

# Stop tracking emissions
model_tracker.stop()
training_emissions = model_tracker.final_emissions

print(f"Model training results:")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score (Macro): {f1_macro:.4f}")
print(f"F1 Score (Micro): {f1_micro:.4f}")
print(f"Hamming Loss: {hl:.4f}")
print(f"Training Carbon Emissions: {training_emissions:.6f} kg CO2eq")


Step 7: Model Optimization (If Needed)

In [ ]:
# If accuracy is below target, try these optimizations
if accuracy < 0.85:
    print("Optimizing model to improve performance...")
    
    # Try different regularization strength
    for C in [0.01, 0.05, 0.1, 0.5, 1.0]:
        opt_tracker = EmissionsTracker()
        opt_tracker.start()
        
        base_model = LogisticRegression(C=C, penalty='l1', solver='liblinear', max_iter=1000)
        multi_model = MultiOutputClassifier(base_model)
        multi_model.fit(X_train, y_train)
        
        y_pred = multi_model.predict(X_test)
        accuracy = accuracy_score(y_test.reshape(-1), y_pred.reshape(-1))
        f1_macro = f1_score(y_test, y_pred, average='macro')
        
        opt_tracker.stop()
        opt_emissions = opt_tracker.final_emissions
        
        print(f"C={C}: Accuracy={accuracy:.4f}, F1={f1_macro:.4f}, Emissions={opt_emissions:.6f}")
        
        if accuracy >= 0.85:
            print(f"Target accuracy achieved with C={C}")
            break


Step 8: Save the Final Model

In [ ]:
import joblib
import sys

# Calculate model size
model_size = sys.getsizeof(multi_model) / (1024 * 1024)  # Size in MB

# Save model components
model_data = {
    'model': multi_model,
    'vectorizer': tfidf_vectorizer,
    'feature_selector': selector,
    'label_columns': label_columns
}

joblib.dump(model_data, 'skin_condition_classifier.pkl')
print(f"Model saved. Size: {model_size:.2f} MB")

# Create a prediction function
def predict_skin_conditions(text, model_data):
    """Predict skin conditions for a new product description"""
    # Preprocess text
    processed = preprocess_text(text)
    
    # Transform to TF-IDF
    X = model_data['vectorizer'].transform([processed])
    
    # Apply feature selection
    X_selected = model_data['feature_selector'].transform(X)
    
    # Track inference emissions
    inference_tracker = EmissionsTracker()
    inference_tracker.start()
    
    # Make prediction
    prediction = model_data['model'].predict(X_selected)[0]
    
    inference_tracker.stop()
    inference_emissions = inference_tracker.final_emissions
    
    # Return predictions and emissions
    result = {
        'predictions': {label: pred for label, pred in zip(model_data['label_columns'], prediction)},
        'emissions': inference_emissions
    }
    
    return result

# Test prediction function
test_text = "This moisturizing cream is perfect for dry, sensitive skin and helps reduce redness."
result = predict_skin_conditions(test_text, model_data)
print("\nSample Prediction:")
for label, value in result['predictions'].items():
    if value == 1:
        print(f"- {label}")
print(f"Inference emissions: {result['emissions']:.6f} kg CO2eq")


Step 9: Carbon Emissions Summary

In [ ]:
# Stop the main tracker
tracker.stop()
total_emissions = tracker.final_emissions

print("\nCarbon Emissions Summary:")
print("--------------------------")
print(f"Total emissions: {total_emissions:.6f} kg CO2eq")
print(f"Training emissions: {training_emissions:.6f} kg CO2eq")
print(f"Average inference emissions: {result['emissions']:.6f} kg CO2eq")
